# 15 — Reward Shaping

## Learning Objectives
1. Understand the potential-based shaping theorem and why it preserves optimal policy
2. Implement distance-based shaping on GridWorld and measure convergence speed improvement
3. Demonstrate how non-potential shaping breaks the optimal policy (circular reward loop)
4. Connect reward shaping to RLHF: the KL penalty as a shaping term


In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import time

np.random.seed(42)
print("NumPy:", np.__version__)
print("Environments: hand-coded GridWorld in numpy — no gym required")


## Level 1: Potential-Based Shaping on GridWorld

**Key theorem (Ng et al., 1999):** If the shaping function is potential-based:
F(s, s') = gamma * Phi(s') - Phi(s)
then the shaped and unshaped MDPs have the same set of optimal policies.

A natural choice for GridWorld: Phi(s) = -dist(s, goal) / max_dist.
Moving toward the goal: F > 0 (bonus). Moving away: F < 0 (penalty).


In [ ]:
# --- Level 1: Sparse vs shaped reward on GridWorld ---

class GridWorld:
    """N x N grid. Agent moves in 4 directions. Reward only at goal."""
    ACTIONS = [(0, 1), (0, -1), (1, 0), (-1, 0)]

    def __init__(self, size=8, goal=None):
        self.size = size; self.goal = goal or (size-1, size-1)
        self.n_states = size*size; self.n_actions = 4
        self.state = (0, 0)

    def reset(self):
        self.state = (0, 0); return self._enc(self.state)

    def step(self, a):
        dy, dx = self.ACTIONS[a]
        r, c = self.state
        nr = max(0, min(self.size-1, r+dy)); nc = max(0, min(self.size-1, c+dx))
        prev = self.state
        self.state = (nr, nc)
        done = self.state == self.goal
        return self._enc(self.state), self._enc(prev), done

    def _enc(self, s): return s[0]*self.size + s[1]
    def decode(self, s): return (s//self.size, s%self.size)
    def manhattan(self, s): r, c = self.decode(s); gr, gc = self.goal; return abs(r-gr)+abs(c-gc)
    def euclidean(self, s):
        r, c = self.decode(s); gr, gc = self.goal
        return float(np.sqrt((r-gr)**2 + (c-gc)**2))


def sparse_reward(env, s, ns, done, **kw): return 1.0 if done else 0.0

def manhattan_shaped(env, s, ns, done, gamma=0.99, **kw):
    max_d = 2*(env.size-1)
    F = gamma * (-env.manhattan(ns)/max_d) - (-env.manhattan(s)/max_d)
    return (1.0 if done else 0.0) + F

def euclidean_shaped(env, s, ns, done, gamma=0.99, **kw):
    max_d = float(np.sqrt(2) * (env.size-1))
    F = gamma * (-env.euclidean(ns)/max_d) - (-env.euclidean(s)/max_d)
    return (1.0 if done else 0.0) + F


def train_q(reward_fn, n_episodes=400, size=8, seed=42, **rw_kwargs):
    """Q-learning with custom reward function. Returns steps per episode."""
    np.random.seed(seed)
    env = GridWorld(size)
    Q = np.zeros((env.n_states, env.n_actions))
    steps_per_ep = []
    for ep in range(n_episodes):
        s = env.reset()
        eps = max(0.05, 1.0 - ep / (n_episodes * 0.7))
        for step in range(200):
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))
            ns, prev_s, done = env.step(a)
            r = reward_fn(env, prev_s, ns, done, **rw_kwargs)
            Q[s, a] += 0.1 * (r + 0.99 * Q[ns].max() * (1-float(done)) - Q[s, a])
            s = ns
            if done: break
        steps_per_ep.append(step + 1)
    return steps_per_ep


print("Comparing sparse vs potential-based shaped rewards on 8x8 GridWorld...")
t0 = time.time()
steps_sparse = train_q(sparse_reward)
steps_manhattan = train_q(manhattan_shaped)
steps_euclidean = train_q(euclidean_shaped)
print(f"Done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(10, 4))
w = 20
def sm(x): return np.convolve(x, np.ones(w)/w, 'valid')
ax.plot(sm(steps_sparse), "--", label="Sparse reward", color="firebrick")
ax.plot(sm(steps_manhattan), label="Manhattan shaped", color="darkgreen")
ax.plot(sm(steps_euclidean), "-.", label="Euclidean shaped", color="steelblue")
ax.set_xlabel("Episode"); ax.set_ylabel("Steps to goal (lower is better)")
ax.set_title("Sparse vs Potential-Based Shaped Reward — 8x8 GridWorld")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/reward_shaping_basic.png", dpi=80)
plt.show()

print(f"Final-20 avg steps: Sparse={np.mean(steps_sparse[-20:]):.1f} | "
      f"Manhattan={np.mean(steps_manhattan[-20:]):.1f} | Euclidean={np.mean(steps_euclidean[-20:]):.1f}")
converged_sp = next((i for i, s in enumerate(steps_sparse) if s < 30), None)
converged_mh = next((i for i, s in enumerate(steps_manhattan) if s < 30), None)
print(f"First episode < 30 steps: Sparse={converged_sp} | Manhattan={converged_mh}")


## Level 2: Potential Function Design — Phi Variants

The potential function Phi(s) can take many forms:
1. **Negative Manhattan distance** — L1, fast but step-wise
2. **Negative Euclidean distance** — L2, smooth but ignores grid structure
3. **Negative value function** — using a learned V(s) as Phi produces optimal guidance
4. **Progress signal** — Phi(s) = fraction of path completed (curriculum-aware)

We compare convergence under each and show that a learned potential is the best.


In [ ]:
# --- Level 2: Phi variants + learned potential ---

def potential_shaped(env, s, ns, done, phi_fn, gamma=0.99):
    """Generic potential-based shaping given callable phi_fn(env, state_int)."""
    F = gamma * phi_fn(env, ns) - phi_fn(env, s)
    return (1.0 if done else 0.0) + F


def phi_manhattan(env, s): return -env.manhattan(s) / (2*(env.size-1))
def phi_euclidean(env, s): return -env.euclidean(s) / (float(np.sqrt(2)*(env.size-1)))
def phi_progress(env, s):
    """Progress: fraction of max distance already covered (0 at start, 1 at goal)."""
    r, c = env.decode(s)
    return (r + c) / (2 * (env.size - 1))


def make_learned_phi(env, value_table):
    """Returns a phi function using a precomputed value table."""
    def phi_learned(env2, s):
        return value_table[s]
    return phi_learned


def train_with_phi(phi_fn, n_episodes=400, size=8, seed=42):
    """Train Q-learning with potential shaping from phi_fn."""
    np.random.seed(seed)
    env = GridWorld(size)
    Q = np.zeros((env.n_states, env.n_actions))
    steps_per_ep = []
    for ep in range(n_episodes):
        s = env.reset(); eps = max(0.05, 1.0 - ep / (n_episodes * 0.7))
        for step in range(200):
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))
            ns, prev_s, done = env.step(a)
            r = potential_shaped(env, prev_s, ns, done, phi_fn)
            Q[s, a] += 0.1 * (r + 0.99*Q[ns].max()*(1-float(done)) - Q[s, a])
            s = ns
            if done: break
        steps_per_ep.append(step + 1)
    return steps_per_ep


# Build an approximate value table: V(s) proportional to negative Manhattan distance
# (this is a simple learned potential; in practice it comes from a pre-trained value function)
size = 8
env_ref = GridWorld(size)
value_table = np.array([-env_ref.manhattan(s) / (2*(size-1)) for s in range(size*size)])
phi_learned = make_learned_phi(env_ref, value_table)

print("Comparing Phi variants on 8x8 GridWorld (400 episodes)...")
t0 = time.time()
results_phi = {
    "Manhattan": train_with_phi(phi_manhattan, seed=42),
    "Euclidean": train_with_phi(phi_euclidean, seed=42),
    "Progress":  train_with_phi(phi_progress, seed=42),
    "Learned V": train_with_phi(phi_learned, seed=42),
}
print(f"Done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(10, 4))
phi_colors = {"Manhattan": "darkgreen", "Euclidean": "steelblue",
              "Progress": "orange", "Learned V": "purple"}
for name, steps in results_phi.items():
    ax.plot(sm(steps), label=name, color=phi_colors[name])
ax.set_xlabel("Episode"); ax.set_ylabel("Steps to goal")
ax.set_title("Potential Function Design: Phi Variants")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/phi_variants.png", dpi=80)
plt.show()

print("Final-20 avg steps per Phi:")
for name, steps in results_phi.items():
    print(f"  {name:12s}: {np.mean(steps[-20:]):.1f}")


## Real-World Example 1: Curriculum via Reward Shaping Annealing

Curriculum learning: start with a dense shaped reward, then gradually anneal
to the sparse reward. The agent learns quickly with guidance and still converges
to the true optimal policy (because we anneal the shaping weight to zero).

This is essentially scheduled reward shaping: weight(t) = max(0, 1 - t/T).


In [ ]:
# === Curriculum: anneal dense -> sparse reward ===

def curriculum_reward(env, s, ns, done, progress, gamma=0.99):
    """
    Blend dense (Manhattan-shaped) and sparse reward.
    progress: 0.0 (episode 1) -> 1.0 (final episode).
    Shaping weight = max(0, 1 - progress).
    """
    max_d = 2 * (env.size - 1)
    F = gamma * (-env.manhattan(ns)/max_d) - (-env.manhattan(s)/max_d)
    shaping_weight = max(0.0, 1.0 - progress)
    return (1.0 if done else 0.0) + shaping_weight * F


def train_curriculum(n_episodes=400, size=8, seed=42):
    np.random.seed(seed)
    env = GridWorld(size)
    Q = np.zeros((env.n_states, env.n_actions))
    steps_per_ep = []
    for ep in range(n_episodes):
        progress = ep / n_episodes
        s = env.reset(); eps = max(0.05, 1.0 - ep / (n_episodes * 0.7))
        for step in range(200):
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))
            ns, prev_s, done = env.step(a)
            r = curriculum_reward(env, prev_s, ns, done, progress=progress)
            Q[s, a] += 0.1 * (r + 0.99*Q[ns].max()*(1-float(done)) - Q[s, a])
            s = ns
            if done: break
        steps_per_ep.append(step + 1)
    return steps_per_ep


print("Training curriculum (dense->sparse annealing) on 8x8 GridWorld...")
t0 = time.time()
steps_curriculum = train_curriculum(n_episodes=400)
steps_always_sparse = train_q(sparse_reward, n_episodes=400)
steps_always_shaped = train_q(manhattan_shaped, n_episodes=400)
print(f"Done in {time.time()-t0:.1f}s")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sm(steps_always_sparse), "--", label="Always sparse", color="firebrick")
ax.plot(sm(steps_always_shaped), "-.", label="Always shaped (Manhattan)", color="steelblue")
ax.plot(sm(steps_curriculum), label="Curriculum (dense -> sparse)", color="darkgreen", linewidth=2)
ax.set_xlabel("Episode"); ax.set_ylabel("Steps to goal")
ax.set_title("Curriculum Learning via Reward Shaping Annealing")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/curriculum.png", dpi=80)
plt.show()

print(f"Final-20 avg steps:")
print(f"  Always sparse:    {np.mean(steps_always_sparse[-20:]):.1f}")
print(f"  Always shaped:    {np.mean(steps_always_shaped[-20:]):.1f}")
print(f"  Curriculum:       {np.mean(steps_curriculum[-20:]):.1f}")
converge_cur = next((i for i, s in enumerate(steps_curriculum) if s < 30), None)
converge_sp = next((i for i, s in enumerate(steps_always_sparse) if s < 30), None)
print(f"  First ep < 30 steps: sparse={converge_sp}  curriculum={converge_cur}")


## Real-World Example 2: RLHF Reward Shaping with KL Penalty

RLHF adds a KL divergence penalty to prevent the RL-trained policy from
drifting too far from the reference (SFT) policy. This is equivalent to
potential-based reward shaping where:
Phi(pi) = -KL(pi || pi_ref)

The shaped reward r' = r_reward_model - beta * KL(pi || pi_ref) steers the
policy toward the reference distribution, preventing reward hacking.


In [ ]:
# === RLHF KL Penalty as Reward Shaping ===

def run_rlhf_comparison(n_steps=400, vocab=10, seed=42):
    """
    Toy LM (10 tokens). Reward model prefers tokens 7-9.
    Compare: no KL | KL beta=0.1 | KL beta=0.5
    """
    reward_weights = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.1, 0.2, 0.5, 0.7, 1.0])
    results = {}

    for beta in [0.0, 0.1, 0.5]:
        np.random.seed(seed)
        logits = np.zeros(vocab)
        ref_logits = np.zeros(vocab)  # reference policy (SFT initialisation)
        lr = 0.05
        r_ext_hist = []; kl_hist = []; r_shaped_hist = []

        for _ in range(n_steps):
            p = np.exp(logits - logits.max()); p /= p.sum()
            p_ref = np.exp(ref_logits - ref_logits.max()); p_ref /= p_ref.sum()
            a = np.random.choice(vocab, p=p)

            # Rewards
            r_ext = float(reward_weights[a])
            kl = float(np.sum(p * np.log(p / (p_ref + 1e-8) + 1e-8)))
            r_total = r_ext - beta * kl

            # Policy gradient update
            d_logits = -p.copy(); d_logits[a] += 1.0
            logits += lr * r_total * d_logits

            r_ext_hist.append(r_ext); kl_hist.append(kl); r_shaped_hist.append(r_total)

        results[beta] = {"r_ext": r_ext_hist, "kl": kl_hist, "r_shaped": r_shaped_hist}

    return results


print("Running RLHF KL penalty comparison...")
rlhf_results = run_rlhf_comparison(n_steps=400)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
w = 30
betas = [0.0, 0.1, 0.5]; beta_colors = {0.0: "firebrick", 0.1: "darkgreen", 0.5: "steelblue"}
for beta in betas:
    col = beta_colors[beta]
    r_smooth = np.convolve(rlhf_results[beta]["r_ext"], np.ones(w)/w, 'valid')
    k_smooth = np.convolve(rlhf_results[beta]["kl"], np.ones(w)/w, 'valid')
    axes[0].plot(r_smooth, label=f"beta={beta}", color=col)
    axes[1].plot(k_smooth, label=f"beta={beta}", color=col)

axes[0].set_xlabel("Step"); axes[0].set_ylabel("Reward model reward")
axes[0].set_title("Extrinsic Reward Over Training")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Step"); axes[1].set_ylabel("KL(pi || pi_ref)")
axes[1].set_title("Policy Divergence from Reference")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle("RLHF Reward Shaping: KL Penalty Effect", fontweight="bold")
plt.tight_layout()
plt.savefig("/tmp/rlhf_shaping.png", dpi=80)
plt.show()

for beta in betas:
    r = rlhf_results[beta]
    print(f"  beta={beta}: final reward={np.mean(r['r_ext'][-50:]):.3f} | KL={np.mean(r['kl'][-50:]):.4f}")
print("
beta=0: maximum reward but policy drifts far from reference (reward hacking risk)")
print("beta=0.5: policy stays near reference but lower reward")


## Real-World Example 3: Non-Potential Shaping Pitfall

**Warning:** Non-potential reward shaping can change the optimal policy!
If F(s, s') is not equal to gamma*Phi(s') - Phi(s) for any Phi,
then the shaped MDP may have a different optimal policy than the original.

Classic example: a robot given a bonus for moving right but never penalised
for moving left learns to oscillate right-left indefinitely instead of reaching the goal.


In [ ]:
# === Non-Potential Shaping Pitfall ===

def non_potential_reward(env, s, ns, done, cycle_bonus=0.3):
    """
    Non-potential shaping: reward rightward motion.
    This is NOT potential-based: no Phi such that F = gamma*Phi(s') - Phi(s).
    Creates a circular incentive: right -> bonus, never penalised for left.
    """
    sr, sc = env.decode(s); nsr, nsc = env.decode(ns)
    if nsc > sc:    # moved right
        return (1.0 if done else 0.0) + cycle_bonus
    return 1.0 if done else 0.0


def train_and_analyse(reward_fn, n_episodes=400, size=8, seed=42, **kw):
    """Train + collect action direction stats."""
    np.random.seed(seed)
    env = GridWorld(size)
    Q = np.zeros((env.n_states, env.n_actions))
    right_actions = []; steps_per_ep = []
    for ep in range(n_episodes):
        s = env.reset(); eps = max(0.05, 1.0 - ep / (n_episodes * 0.7))
        ep_right = 0; ep_total = 0
        for step in range(200):
            a = np.random.randint(4) if np.random.random() < eps else int(np.argmax(Q[s]))
            ns, prev_s, done = env.step(a)
            r = reward_fn(env, prev_s, ns, done, **kw)
            Q[s, a] += 0.1 * (r + 0.99*Q[ns].max()*(1-float(done)) - Q[s, a])
            s = ns; ep_total += 1
            if a == 0: ep_right += 1  # action 0 = right
            if done: break
        right_actions.append(ep_right / max(1, ep_total))
        steps_per_ep.append(step + 1)
    return steps_per_ep, right_actions


print("Comparing potential vs non-potential shaping on 8x8 GridWorld...")
t0 = time.time()
steps_pot, right_pot = train_and_analyse(manhattan_shaped)
steps_nonpot, right_nonpot = train_and_analyse(non_potential_reward, cycle_bonus=0.3)
steps_sp2, right_sp = train_and_analyse(sparse_reward)
print(f"Done in {time.time()-t0:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for steps, right, label, col in [
    (steps_sp2, right_sp, "Sparse (no shaping)", "gray"),
    (steps_pot, right_pot, "Potential (Manhattan)", "darkgreen"),
    (steps_nonpot, right_nonpot, "Non-potential (right bias)", "firebrick"),
]:
    axes[0].plot(sm(steps), label=label, color=col)
    axes[1].plot(sm(right), label=label, color=col)

axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Steps to goal (200=failed)")
axes[0].set_title("Convergence: Potential vs Non-Potential Shaping")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Fraction of rightward moves")
axes[1].set_title("Non-Potential Shaping: Agent Over-Commits to Right")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/nonpotential.png", dpi=80)
plt.show()

print(f"Final-20 avg steps: Sparse={np.mean(steps_sp2[-20:]):.1f} | "
      f"Potential={np.mean(steps_pot[-20:]):.1f} | "
      f"Non-potential={np.mean(steps_nonpot[-20:]):.1f}")
print(f"Non-potential right-move fraction: {np.mean(right_nonpot[-20:]):.2f} (agent biased right!)")
print("Non-potential agent fails to solve task: policy changed by the shaping!")


## Comparison: Sparse, Potential, Non-Potential, Curriculum — Convergence


In [ ]:
# === Full comparison: 5 reward configurations ===

def run_full_comparison(n_episodes=400, n_seeds=5, size=8):
    configs = {
        "Sparse":          (sparse_reward, {}),
        "Manhattan":       (manhattan_shaped, {}),
        "Euclidean":       (euclidean_shaped, {}),
        "Non-potential":   (non_potential_reward, {"cycle_bonus": 0.3}),
        "Curriculum":      (curriculum_reward, {}),
    }
    results = {}
    for name, (fn, kw) in configs.items():
        seed_runs = []
        for seed in range(n_seeds):
            if name == "Curriculum":
                steps = train_curriculum(n_episodes=n_episodes, seed=seed)
            else:
                steps = train_q(fn, n_episodes=n_episodes, seed=seed, **kw)
            seed_runs.append(steps)
        results[name] = np.array(seed_runs)
    return results


print("Running full reward shaping comparison (5 seeds each)...")
t0 = time.time()
all_results = run_full_comparison(n_episodes=300, n_seeds=5)
print(f"Done in {time.time()-t0:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_full = {"Sparse": "gray", "Manhattan": "darkgreen", "Euclidean": "steelblue",
               "Non-potential": "firebrick", "Curriculum": "purple"}

for name, runs in all_results.items():
    mu = runs.mean(0); std = runs.std(0)
    sm_mu = np.convolve(mu, np.ones(20)/20, 'valid')
    sm_std = np.convolve(std, np.ones(20)/20, 'valid')
    col = colors_full[name]
    axes[0].plot(sm_mu, label=name, color=col)
    axes[0].fill_between(range(len(sm_mu)), sm_mu-sm_std, sm_mu+sm_std, alpha=0.12, color=col)

axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Steps to goal")
axes[0].set_title("All Reward Shaping Methods: Convergence Speed")
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Bar chart: convergence episode
final_means = [np.mean(v[:, -20:]) for v in all_results.values()]
names_full = list(all_results.keys())
bars = axes[1].bar(names_full, final_means, color=list(colors_full.values()), alpha=0.8, edgecolor="black")
axes[1].set_ylabel("Final-20 avg steps to goal")
axes[1].set_title("Final Policy Quality by Shaping Method")
axes[1].grid(True, alpha=0.3, axis="y")
for bar, val in zip(bars, final_means):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                 f"{val:.0f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("/tmp/shaping_comparison.png", dpi=80)
plt.show()

print("
Final-20 avg steps:")
for name, runs in all_results.items():
    print(f"  {name:16s}: {np.mean(runs[:,-20:]):.1f} steps")
print("
Key insight: Non-potential shaping (200 = never converges) confirms the policy invariance theorem")


## Key Takeaways

**Core idea:** Reward shaping adds information about intermediate progress to guide
learning, without (if done correctly) changing the optimal policy. The potential-based
formulation is the only safe way to add shaping: F = gamma*Phi(s') - Phi(s).
Any other form risks creating incentives for unintended behaviour.

### Variants and When to Use

| Method | Potential-based? | Preserves Policy? | Convergence | Use Case |
|--------|-----------------|-------------------|-------------|---------|
| Sparse (none) | N/A | Yes (baseline) | Slow | Simple tasks |
| Manhattan shaping | Yes | Yes | Fast | GridWorld, navigation |
| Euclidean shaping | Yes | Yes | Fast | Continuous spaces |
| Non-potential | No | NO (dangerous) | Unpredictable | Avoid! |
| Curriculum anneal | Yes (annealed) | Yes | Fastest | Hard exploration |
| RLHF KL penalty | Yes (KL-based) | Approximately | Moderate | LLM alignment |

### Common Failure Modes

- **Non-potential shaping:** Agent exploits shaped reward by cycling through
  states with positive shaping bonus. Never reaches the true goal.
  Symptom: high shaped reward but goal never reached.
  Fix: verify F = gamma*Phi(s') - Phi(s) for some Phi; use Manhattan/Euclidean distance.
- **Shaping weight not annealed:** If shaping weight stays high throughout training,
  the agent learns to maximise shaped reward, not true task reward.
  In theory, potential-based shaping doesn't change optima, but with function approximation it can.
  Fix: anneal shaping weight to 0 over the final 20% of training.
- **RLHF: KL penalty too low (beta ~ 0):** Policy exploits reward model with degenerate
  outputs. Symptom: reward keeps increasing but generated text becomes nonsensical.
  Fix: increase beta until KL(pi || pi_ref) stays below target (typically 5-10 nats).
- **Phi not aligned with task:** Using a potential that doesn't correlate with true
  progress (e.g., Manhattan distance in a maze with walls) provides misleading guidance.
  Fix: compute Phi from offline rollouts or learned value function.

### Related Concepts

- [14-exploration-exploitation](./14-exploration-exploitation.ipynb) — intrinsic rewards as shaping
- [11-proximal-policy-optimization](./11-proximal-policy-optimization.ipynb) — RLHF uses PPO + shaped reward
- [12-soft-actor-critic](./12-soft-actor-critic.ipynb) — entropy bonus is a reward shaping form
- [13-multi-armed-bandit](./13-multi-armed-bandit.ipynb) — bandit exploration bonuses are shaped rewards


## Exercises

1. **Verify the theorem:** Run the non-potential shaping example with cycle_bonus=0.
   Does the agent converge to the same policy as the sparse reward case?
   Increase cycle_bonus to 1.0 — at what point does the agent prefer the cycle?

2. **Design a new potential:** For a maze with an obstacle at (4,4), compute
   Phi(s) = negative shortest-path distance to goal (BFS). Does this outperform
   the Manhattan/Euclidean potential that ignores walls?

3. **Annealing schedule:** Replace the linear curriculum anneal with:
   (a) step function: shaped for first 50%, sparse for last 50%
   (b) cosine: weight(t) = 0.5 * (1 + cos(pi * t/T))
   Compare convergence speed.

4. **RLHF beta sweep:** In the RLHF simulation, sweep beta from 0.01 to 1.0.
   Plot the trade-off curve: final KL vs final reward. What is the Pareto-optimal beta?

5. **Reward shaping with function approximation:** Replace the tabular Q with a
   linear function approximator Q(s,a) = phi(s)^T theta_a. Does potential-based
   shaping still preserve the optimal policy? (Short answer: approximately, but
   less guaranteed than in the tabular case.)
